# Lecture 3 — Stochastic optimization

**Week 2 · Day 3 · 45 min**

> **Headline.** When $n$ is large, use a *sample* of the gradient. Being approximately
> right very often beats being exactly right occasionally.

Every gradient so far cost a full pass over the data. For $n = 10^6$ that is one step per
sweep — and you may need thousands of steps. Today we stop insisting on the exact
gradient, and in exchange we get to take thousands of steps per sweep.

This is not a compromise made in the corner of the field. **It is how essentially every
modern machine-learning model is trained.**

**By the end of this lecture you can:**

1. show a mini-batch gradient is unbiased and that its variance falls like $1/b$;
2. explain why a constant step size converges to a *noise floor* and not to the optimum;
3. state the Robbins–Monro conditions and why both are needed;
4. explain what Adam adapts to, and connect it to day 1's conditioning.

**You implement this afternoon:** `BatchObjective` on `GLMLoss`, then `SGD` and `Adam`.

### Pacing

Target **35 min** of core material, hard cap **45 min**. Sections marked
*(cut first)* are the ones to drop if you are running behind; everything else is
load-bearing for the labwork. **The times below already include showing and discussing
the figures** — each figure is produced by the code cell above it, so run the notebook
once before the session.


> Figure 3 (the Robbins–Monro conditions) is the one to drop if §2 is running long — the
> floor in Figure 2 is the load-bearing idea, and the labwork only needs that.


| § | Section | min |
|---|---|---|
| 1 | The finite-sum structure  — *Figure 1* | 10 |
| 2 | What the noise costs: the noise floor  — *Figures 2–3* | 12 |
| 3 | Adam: a step size per coordinate  — *Figure 4* | 11 |
| 4 | Epochs, shuffling, seeds  *(cut first)* | 5 |
| 5 | Today's labs | 2 |
| | **total** | **40** |
| | **core only** | **35** |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=6, suppress=True)

# One consistent look for every figure in the lecture.
plt.rcParams.update({
    "figure.dpi": 110,
    "font.size": 9,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

# If this import fails:   pip install matplotlib

rng = np.random.default_rng(3)

---

## 1. The finite-sum structure

Every loss this week has the same shape, and we have not yet exploited it:

$$f(w) \;=\; \frac{1}{n}\sum_{i=1}^{n} f_i(w), \qquad f_i(w) = \varphi(x_i^\top w,\, y_i)$$

so the gradient is also an average:

$$\nabla f(w) \;=\; \frac{1}{n}\sum_{i=1}^{n} \nabla f_i(w)$$

**An average is exactly the kind of thing you can estimate from a sample.** Draw a
random subset $B \subset \{1,\dots,n\}$ of size $b$ and compute

$$g_B(w) \;=\; \frac{1}{b}\sum_{i \in B} \nabla f_i(w)$$

Cost $O(b)$ instead of $O(n)$. For $b = 32$ and $n = 10^6$, that is **30 000 times
cheaper per step**.

### It is unbiased

If each index is drawn uniformly, then for any fixed $w$

$$\mathbb{E}[g_B(w)] \;=\; \frac{1}{b}\sum_{i \in B}\mathbb{E}[\nabla f_i(w)]
\;=\; \frac{1}{b} \cdot b \cdot \frac{1}{n}\sum_{j=1}^n \nabla f_j(w) \;=\; \nabla f(w)$$

**On average it points the right way.** That is the entire justification for the method.
It is wrong at every individual step and right in expectation, and over many steps the
errors average out while the signal accumulates.

### Its variance falls like $1/b$

For independent draws, the variance of a mean of $b$ terms is $\sigma^2/b$ where
$\sigma^2$ is the per-sample gradient variance. So the noise in the direction shrinks
like $1/\sqrt{b}$ — with **diminishing returns**: going from $b=1$ to $b=32$ cuts the
noise by $5.7\times$, from $32$ to $1024$ by another $5.7\times$ at $32\times$ the cost.
This is why moderate batch sizes are the norm.

In [ ]:
# A concrete GLM: logistic regression on synthetic data.
n, p = 5000, 10
X = rng.normal(size=(n, p))
w_true = rng.normal(size=p)
y = (rng.random(n) < 1 / (1 + np.exp(-X @ w_true))).astype(float)

def sigmoid(z):
    """Stable logistic function: no overflow for large |z|."""
    return np.where(z >= 0, 1.0 / (1.0 + np.exp(-np.abs(z))),
                    np.exp(-np.abs(z)) / (1.0 + np.exp(-np.abs(z))))

def full_gradient(w):
    return X.T @ (sigmoid(X @ w) - y) / n

def batch_gradient(w, idx):
    Xb, yb = X[idx], y[idx]
    return Xb.T @ (sigmoid(Xb @ w) - yb) / len(idx)

w = rng.normal(size=p) * 0.3
g_full = full_gradient(w)

print(f"{'batch size b':>13} {'bias':>12} {'std of error':>14} {'predicted ~1/sqrt(b)':>22}")
for b in [1, 4, 16, 64, 256, 1024]:
    errs = np.array([batch_gradient(w, rng.choice(n, b, replace=False)) - g_full
                     for _ in range(400)])
    print(f"{b:13d} {np.linalg.norm(errs.mean(axis=0)):12.5f} "
          f"{errs.std(axis=0).mean():14.5f} {1/np.sqrt(b):22.5f}")

Two things to read off that table. The **bias column stays near zero** at every batch
size — the estimator is unbiased, as derived. The **error column falls like
$1/\sqrt{b}$**, tracking the last column. Noise, not bias, is what we are trading against
cost.

And the limiting case is a useful sanity check: with $b = n$ the "estimate" is the exact
gradient, so **SGD with `batch_size = n` must reduce to gradient descent**. That is a test
you will write this afternoon.

In [ ]:
# What "unbiased with variance sigma^2/b" looks like. Two coordinates of the gradient,
# 400 independent mini-batches each.
fig, ax = plt.subplots(1, 2, figsize=(11.5, 4.3))

cols = {1: "tab:red", 16: "tab:orange", 256: "tab:blue"}
for b, col in cols.items():
    sample = np.array([batch_gradient(w, rng.choice(n, b, replace=False))
                       for _ in range(400)])
    ax[0].scatter(sample[:, 0], sample[:, 1], s=7, alpha=0.35, color=col,
                  label=f"b = {b}", zorder=3)

ax[0].plot(g_full[0], g_full[1], "*", color="gold", ms=20, mec="k", mew=1.0, zorder=6)
ax[0].annotate("true $\\nabla f$", xy=(g_full[0], g_full[1]), xytext=(18, 16),
               textcoords="offset points", fontsize=9, fontweight="bold",
               arrowprops=dict(arrowstyle="->", lw=1.2))
ax[0].axhline(g_full[1], color="0.4", lw=0.8, ls=":")
ax[0].axvline(g_full[0], color="0.4", lw=0.8, ls=":")
ax[0].set_xlabel("gradient, coordinate 1"); ax[0].set_ylabel("gradient, coordinate 2")
ax[0].set_title("Every cloud is centred on the truth;\nonly its width changes", fontsize=9)
ax[0].legend(fontsize=8, markerscale=2.2, loc="upper left")

# --- right: the 1/sqrt(b) law, over six batch sizes.
bs = np.array([1, 4, 16, 64, 256, 1024])
stds, biases = [], []
for b in bs:
    e = np.array([batch_gradient(w, rng.choice(n, b, replace=False)) - g_full
                  for _ in range(400)])
    stds.append(e.std(axis=0).mean())
    biases.append(np.linalg.norm(e.mean(axis=0)))

ax[1].loglog(bs, stds, "o-", color="tab:blue", lw=1.8, ms=7, label="measured noise (std)")
ax[1].loglog(bs, stds[0] / np.sqrt(bs), "--", color="0.35", lw=1.4,
             label=r"$\propto 1/\sqrt{b}$")
ax[1].loglog(bs, biases, "s-", color="crimson", lw=1.5, ms=6,
             label="measured bias (400 batches)")
ax[1].loglog(bs, np.array(stds) / np.sqrt(400), ":", color="crimson", lw=1.4,
             label=r"Monte-Carlo error $\sigma_b/\sqrt{400}$")
ax[1].set_xlabel("batch size $b$"); ax[1].set_ylabel("error in the gradient estimate")
ax[1].set_title("Noise falls like $1/\\sqrt{b}$; bias stays at zero", fontsize=9)
ax[1].legend(fontsize=8)

plt.tight_layout()
plt.show()

print(f"noise at b=1    : {stds[0]:.4f}")
print(f"noise at b=1024 : {stds[-1]:.4f}   ->  {stds[0]/stds[-1]:.1f}x smaller "
      f"for {bs[-1]}x the cost per step")

**Figure 1 — the two facts that make the whole method work.**

*Left:* each dot is one mini-batch gradient, for three batch sizes. The three clouds sit
on **the same centre** — the true gradient (gold star). That is unbiasedness, and it is
what lets the errors cancel over many steps instead of accumulating. What changes with $b$
is only the *width* of the cloud.

Notice how wide the $b = 1$ cloud is. A single-sample gradient frequently points into a
completely different quadrant from the true one — individual steps are often not merely
inaccurate but qualitatively wrong. SGD works anyway, because it is the *average over
steps* that has to be right, not any one step.

*Right:* the same experiment measured. The blue line is the observed noise, the dashed
line is $1/\sqrt{b}$ — they are parallel, so the law holds.

The red line needs care. It is the *measured* bias, and it is not flat — but that is not
evidence of real bias. We estimated it by averaging 400 batches, and the error of such an
average is itself about $\sigma_b/\sqrt{400}$, the red dotted line. The measurement sits
on that line at every $b$, which is exactly what a **true bias of zero** looks like
through a finite sample. Note that it stays roughly $20\times$ below the noise line
throughout: what limits SGD is variance, not bias.

> **Read the diminishing returns off the log-log slope.** A slope of $-1/2$ means
> quadrupling the batch halves the noise. The cost, meanwhile, quadruples. That is why
> nobody uses $b = 10^4$: you pay $100\times$ for a $10\times$ cleaner gradient, when you
> could have taken 100 noisier steps instead.

---

## 2. What the noise costs: the noise floor

Here is the central phenomenon. Run SGD with a **constant** step $\alpha$:

$$w_{k+1} = w_k - \alpha\, g_{B_k}(w_k)$$

Near the optimum $\nabla f(w^\star) = 0$, but the *sampled* gradient is not zero — the
individual $\nabla f_i(w^\star)$ do not vanish, only their average does. So the iterate
never settles. It reaches a neighbourhood of $w^\star$ and then rattles around inside it
forever.

The size of that neighbourhood — the **noise floor** — scales like $\alpha$:

$$\mathbb{E}\big[f(w_k) - f(w^\star)\big] \;\xrightarrow{k \to \infty}\; O\!\left(\frac{\alpha\sigma^2}{\mu}\right)$$

**Shrink the step, shrink the floor** — and slow down the approach to it. That trade-off
is what makes schedules necessary.

In [ ]:
def softplus(z):
    """Stable log(1 + exp(z)) -- the day-1 trap, and we are about to need it."""
    return np.maximum(z, 0) + np.log1p(np.exp(-np.abs(z)))


def loss(w, Xm=None, ym=None):
    Xm = X if Xm is None else Xm
    ym = y if ym is None else ym
    z = Xm @ w
    return float(np.mean(softplus(z) - ym * z))


def sgd(w0, lr, n_epochs, b, decay=0.0, seed=0):
    """Plain mini-batch SGD. Step at epoch k is lr / (1 + decay * k)."""
    gen = np.random.default_rng(seed)
    w = w0.copy()
    losses = []
    for epoch in range(n_epochs):
        step = lr / (1 + decay * epoch)
        order = gen.permutation(len(y))             # reshuffle every epoch
        for start in range(0, len(y), b):
            idx = order[start:start + b]
            w = w - step * batch_gradient(w, idx)
        losses.append(loss(w))
    return w, np.array(losses)


# The optimum, found accurately with full-batch descent, so we can measure the EXCESS
# loss f(w) - f*. Measuring the raw loss hides the effect entirely.
w_star = np.zeros(p)
for _ in range(20000):
    w_star -= 2.0 * full_gradient(w_star)
f_star = loss(w_star)
print(f"reference optimum f* = {f_star:.8f}  (|grad| = {np.linalg.norm(full_gradient(w_star)):.2e})\n")

print("constant step -- the noise FLOOR is proportional to alpha:")
prev = None
for lr in [0.4, 0.2, 0.1, 0.05]:
    _, losses = sgd(np.zeros(p), lr, 80, b=32)
    excess = losses[-20:].mean() - f_star
    ratio = "" if prev is None else f"   (previous / this = {prev / excess:.2f})"
    print(f"  lr = {lr:5.2f}   excess loss f - f* = {excess:.3e}{ratio}")
    prev = excess

In [ ]:
# The noise floor, and what "orbiting" actually looks like.
def sgd_track(w0, lr, n_epochs, b, decay=0.0, seed=0):
    """SGD that also records w at the end of every epoch."""
    gen = np.random.default_rng(seed)
    w, losses, ws = w0.copy(), [], []
    for epoch in range(n_epochs):
        step = lr / (1 + decay * epoch)
        order = gen.permutation(len(y))
        for start in range(0, len(y), b):
            w = w - step * batch_gradient(w, order[start:start + b])
        losses.append(loss(w)); ws.append(w.copy())
    return np.array(losses), np.array(ws)


fig, ax = plt.subplots(1, 2, figsize=(11.5, 4.3))

# --- left: a floor per step size, plus the decaying schedule that escapes them all.
for lr, col in [(0.4, "crimson"), (0.2, "tab:orange"), (0.1, "tab:green"), (0.05, "tab:blue")]:
    ls, _ = sgd_track(np.zeros(p), lr, 120, b=32)
    floor = ls[-20:].mean() - f_star
    ax[0].semilogy(ls - f_star, color=col, lw=1.4, label=f"constant $\\alpha$ = {lr}")
    ax[0].axhline(floor, color=col, lw=0.9, ls=":")

ls_dec, _ = sgd_track(np.zeros(p), 0.4, 120, b=32, decay=0.2)
ax[0].semilogy(ls_dec - f_star, color="k", lw=2.2, label=r"decaying $\alpha_k=\alpha_0/(1+0.2k)$")
ax[0].set_xlabel("epoch"); ax[0].set_ylabel("excess loss  $f(w_k) - f^\\star$")
ax[0].set_title("Each constant step stalls at its own floor", fontsize=9)
ax[0].legend(fontsize=7.5, loc="upper right")

# --- right: the iterates themselves, once the floor is reached.
_, ws_c = sgd_track(np.zeros(p), 0.4, 200, b=32)
_, ws_d = sgd_track(np.zeros(p), 0.4, 200, b=32, decay=0.2)
ax[1].plot(w_star[0], w_star[1], "*", color="gold", ms=20, mec="k", mew=1.0, zorder=2,
           label="$w^\\star$")
ax[1].scatter(ws_c[-120:, 0], ws_c[-120:, 1], s=16, color="crimson", alpha=0.5,
              label="constant $\\alpha$ — last 120 epochs", zorder=3)
ax[1].scatter(ws_d[-120:, 0], ws_d[-120:, 1], s=16, color="tab:blue", alpha=0.8,
              label="decaying $\\alpha$ — last 120 epochs", zorder=5)
sc, sd = ws_c[-120:, :2].std(axis=0).mean(), ws_d[-120:, :2].std(axis=0).mean()
ax[1].set_xlabel("$w_1$"); ax[1].set_ylabel("$w_2$")
ax[1].set_title(f"The constant step never stops moving\n"
                f"(spread: {sc:.3f} constant vs {sd:.4f} decaying — {sc/sd:.0f}x tighter)",
                fontsize=9)
ax[1].legend(fontsize=8, loc="upper left")

plt.tight_layout()
plt.show()

print(f"spread of the last 120 iterates (constant): {ws_c[-120:].std(axis=0)[:2]}")
print(f"spread of the last 120 iterates (decaying): {ws_d[-120:].std(axis=0)[:2]}")

**Figure 2 — the noise floor is a real place, and the iterate lives in it.**

*Left:* four constant step sizes. Every curve drops quickly, then goes **flat** — each at
its own height, marked by the dotted line, and smaller $\alpha$ gives a lower floor. No
amount of extra computation moves a flat curve down; the run has converged to a
*distribution*, not to a point. The black curve uses a decaying schedule and keeps
descending past all four floors.

*Right:* the same thing in parameter space — the last 120 epochs of two runs, in the plane
of the first two coordinates. The constant-step iterates (red) form a permanent cloud
around $w^\star$: the run is orbiting and will orbit forever. The decaying-step iterates
(blue) have collapsed onto the optimum.

> **This is the single most useful thing to recognize in a real training curve.** A loss
> that has gone flat but noisy has not "finished training" — it has hit the floor set by
> the step size. Dropping the learning rate is what moves it, and that is precisely what a
> learning-rate schedule automates.

Each halving of the step buys a substantial drop in the excess loss at the plateau. (The
theoretical floor is proportional to $\alpha$; the measured ratios exceed $2$ because at
the larger steps we are not purely floor-limited yet. The direction of the effect is the
point.) The iterate is not converging — it is orbiting.

### The fix: let the step go to zero, but not too fast

**Robbins–Monro conditions.** A schedule $\alpha_k$ gives almost-sure convergence if

$$\sum_{k} \alpha_k = \infty \qquad\text{and}\qquad \sum_{k} \alpha_k^2 < \infty$$

Both matter, and each rules out a distinct failure:

- $\sum \alpha_k = \infty$ — **the steps must not shrink so fast that you stall.** If
  $\alpha_k = 2^{-k}$ the total distance you can ever travel is bounded, and you converge
  to wherever you happened to be, not to $w^\star$.
- $\sum \alpha_k^2 < \infty$ — **the accumulated noise must be finite**, so the floor
  actually closes to zero.

$\alpha_k = \alpha_0/(1+ck)$ satisfies both ($\sum 1/k$ diverges, $\sum 1/k^2$ converges).
A constant step satisfies the first and fails the second — which is precisely the noise
floor we just measured.

In [ ]:
# The two Robbins-Monro conditions, made visible. Three schedules, 2000 steps.
k = np.arange(1, 5001)
sched = [
    ("constant  $\\alpha_k = 0.1$",           np.full_like(k, 0.1, dtype=float), "crimson"),
    ("$\\alpha_k = 0.1/(1+0.01k)$",           0.1 / (1 + 0.01 * k),              "tab:green"),
    ("geometric  $0.1\\cdot 2^{-k/50}$",      0.1 * 2.0 ** (-k / 50.0),          "tab:blue"),
]

fig, ax = plt.subplots(1, 3, figsize=(13.5, 3.9))
for name, a, col in sched:
    ax[0].loglog(k, a, color=col, lw=1.9, label=name)
    ax[1].plot(k, np.cumsum(a), color=col, lw=1.9)
    ax[2].plot(k, np.cumsum(a ** 2), color=col, lw=1.9)

ax[0].set_title(r"the schedule $\alpha_k$", fontsize=9)
ax[0].set_xlabel("step $k$"); ax[0].set_ylabel(r"$\alpha_k$")
ax[0].legend(fontsize=7.5, loc="lower left")

# Linear x with a log y: a DIVERGING sum keeps rising, a CONVERGING one goes flat.
for a_, ttl in [(ax[1], r"$\sum\alpha_k$  — must DIVERGE" "\n" r"(total distance you may travel)"),
                (ax[2], r"$\sum\alpha_k^2$  — must CONVERGE" "\n" r"(total noise you accumulate)")]:
    a_.set_yscale("log"); a_.set_xlabel("step $k$"); a_.set_title(ttl, fontsize=9)

ax[1].annotate("FLAT = stalls:\nit can never travel\nfurther than 7.2",
               xy=(2600, 7.2), xytext=(1400, 1.1), fontsize=8, color="tab:blue",
               bbox=dict(fc="white", ec="tab:blue", lw=0.8, alpha=0.95),
               arrowprops=dict(arrowstyle="->", color="tab:blue", lw=1.1))
ax[1].annotate("still rising ✓", xy=(4300, 39), xytext=(0, 12),
               textcoords="offset points", fontsize=8, color="tab:green", ha="center")

ax[2].annotate("STILL RISING = the\nnoise never settles",
               xy=(3500, 35), xytext=(900, 3.0), fontsize=8, color="crimson",
               bbox=dict(fc="white", ec="crimson", lw=0.8, alpha=0.95),
               arrowprops=dict(arrowstyle="->", color="crimson", lw=1.1))
ax[2].annotate("flat ✓", xy=(4300, 1.0), xytext=(0, 11), textcoords="offset points",
               fontsize=8, color="tab:green", ha="center")

plt.tight_layout()
plt.show()

for name, a, _ in sched:
    lbl = name.split("  ")[0].replace("$", "")
    print(f"{lbl:28s}  sum a = {np.cumsum(a)[-1]:9.2f}   sum a^2 = {np.cumsum(a**2)[-1]:8.3f}")

**Figure 3 — why the conditions come in a pair.**

Each condition kills one failure mode, and the middle and right panels show exactly which.

| schedule | $\sum \alpha_k$ | $\sum \alpha_k^2$ | verdict |
|---|---|---|---|
| constant $\alpha$ | diverges ✓ | **diverges ✗** | never stops rattling — the floor of Figure 2 |
| $\alpha_0/(1+ck)$ | diverges ✓ | converges ✓ | **both satisfied — converges** |
| geometric decay | **converges ✗** | converges ✓ | stalls early, wherever it happens to be |

Both right-hand panels use a **linear** $k$ axis and a log value axis, so the shape is what
to read: a sum that keeps climbing diverges, a sum that goes flat converges.

Read the middle panel as a *travel budget*: $\sum\alpha_k$ is roughly how far the iterate
could ever move. For the blue schedule that budget is finite and tiny, so the run freezes
at whatever point it has reached — it will look beautifully converged and be in the wrong
place. Read the right panel as *accumulated noise*: for the constant step it grows without
bound, which is why that run never settles.

The middle schedule is the only one that keeps both promises: it can still reach any point
(the sum diverges, though only logarithmically — which is why SGD's tail is slow), while
the injected noise totals to something finite.

> The blue curve is $2^{-k/50}$ rather than the text's $2^{-k}$, purely so that it is
> visible on the plot at all. The real $2^{-k}$ is *more* extreme — its total travel
> budget is $0.1$ — and fails the first condition even harder.

In [ ]:
_, const = sgd(np.zeros(p), 0.4, 150, b=32, decay=0.0)
_, decay = sgd(np.zeros(p), 0.4, 150, b=32, decay=0.2)

print(f"{'epoch':>7} {'constant: f - f*':>20} {'decaying: f - f*':>20}")
for e in [10, 30, 60, 100, 149]:
    print(f"{e:7d} {const[e] - f_star:20.3e} {decay[e] - f_star:20.3e}")

print("\nhow much it is still rattling (std over the last 20 epochs):")
print(f"  constant : {const[-20:].std():.2e}   <- orbiting, and it will orbit forever")
print(f"  decaying : {decay[-20:].std():.2e}   <- settling")

---

## 3. Adam: a step size per coordinate

Momentum (day 2) smooths the *direction*. Adam goes further and adapts the **step size
separately for every coordinate**, using the running magnitude of that coordinate's
gradient.

Keep two exponential moving averages:

$$m_k = \beta_1 m_{k-1} + (1-\beta_1) g_k \qquad \text{(first moment — the direction)}$$
$$v_k = \beta_2 v_{k-1} + (1-\beta_2) g_k^{\odot 2} \qquad \text{(second moment — the scale)}$$

Both start at zero, so both are biased toward zero early on. Correct for it:

$$\hat m_k = \frac{m_k}{1-\beta_1^k}, \qquad \hat v_k = \frac{v_k}{1-\beta_2^k}$$

and step:

$$\boxed{\;w_{k+1} = w_k - \alpha\,\frac{\hat m_k}{\sqrt{\hat v_k} + \varepsilon}\;}$$

Defaults $\beta_1 = 0.9$, $\beta_2 = 0.999$, $\varepsilon = 10^{-8}$.

### What is it actually doing?

Look at the update: the gradient is **divided by its own typical magnitude**. A coordinate
with consistently large gradients gets a small multiplier; one with small gradients gets a
large multiplier. The step becomes roughly scale-free in each coordinate.

**Adam auto-standardizes the per-coordinate scale.** Which is day 1's conditioning problem
— and note it is now being solved on the *optimizer* side rather than by rescaling the
data. Two responses to the same disease.

At the very first step, $\hat m_1 = g_1$ and $\sqrt{\hat v_1} = |g_1|$, so the update is
$\alpha \cdot g_1/|g_1| = \alpha\,\mathrm{sign}(g_1)$ — a fixed-size step regardless of
gradient magnitude. A good test, and a good way to remember what Adam is.

In [ ]:
# The first Adam step is lr * sign(g), whatever the scale of g.
for scale in [1e-3, 1.0, 1e3]:
    g = np.array([scale, -scale, 2 * scale])
    m = (1 - 0.9) * g
    v = (1 - 0.999) * g ** 2
    step = 0.001 * (m / (1 - 0.9)) / (np.sqrt(v / (1 - 0.999)) + 1e-8)
    print(f"|g| ~ {scale:8.0e}  ->  first step = {step}")
print("\nIdentical steps across six orders of magnitude of gradient. That is the point.")

In [ ]:
def adam(w0, lr, n_epochs, b, beta1=0.9, beta2=0.999, eps=1e-8, seed=0):
    gen = np.random.default_rng(seed)
    w = w0.copy()
    m, v, t = np.zeros_like(w), np.zeros_like(w), 0
    losses = []
    for _ in range(n_epochs):
        order = gen.permutation(n)
        for start in range(0, n, b):
            idx = order[start:start + b]
            g = batch_gradient(w, idx)
            t += 1
            m = beta1 * m + (1 - beta1) * g
            v = beta2 * v + (1 - beta2) * g ** 2
            w = w - lr * (m / (1 - beta1 ** t)) / (np.sqrt(v / (1 - beta2 ** t)) + eps)
        losses.append(loss(w))
    return w, np.array(losses)


# The badly scaled problem: one feature recorded in units 1000x larger than the rest.
X_orig = X.copy()
X = X_orig * np.concatenate([[1000.0], np.ones(p - 1)])

w_opt = w_true.copy()
w_opt[0] /= 1000.0                      # the same model, expressed in the new units
print(f"one feature scaled by 1000x  ->  kappa = {np.linalg.cond(X.T @ X / len(y)):.2e}")
print(f"  loss at w = 0 : {loss(np.zeros(p)):.6f}")
print(f"  loss at w*    : {loss(w_opt):.6f}   <- the target\n")

# SGD is trapped. Any step big enough to move the small-scale coordinates is unstable
# on the large-scale one, because stability needs alpha < 2/L and L is now enormous.
for lr in [1e-2, 1e-4, 1e-6, 1e-7]:
    _, l = sgd(np.zeros(p), lr, 10, b=32)
    print(f"  SGD lr = {lr:.0e}  ->  loss after 10 epochs = {l[-1]:12.6f}"
          f"{'   (climbing: unstable)' if l[-1] > l[0] else ''}")

_, l_sgd  = sgd(np.zeros(p), 1e-7, 40, b=32)     # the largest step that is stable
_, l_adam = adam(np.zeros(p), 1e-3, 40, b=32)    # Adam is unbothered by the scale

print(f"\n{'epoch':>7} {'SGD (lr 1e-7)':>16} {'Adam (lr 1e-3)':>16}")
for e in [0, 4, 9, 19, 39]:
    print(f"{e:7d} {l_sgd[e]:16.6f} {l_adam[e]:16.6f}")
print(f"{'target':>7} {loss(w_opt):16.6f} {loss(w_opt):16.6f}")

X = X_orig  # restore

In [ ]:
# What Adam is compensating for, on the same badly scaled problem.
X_bad = X_orig * np.concatenate([[1000.0], np.ones(p - 1)])
X = X_bad                               # the helper closures read the global X
w_tgt = w_true.copy(); w_tgt[0] /= 1000.0
target = loss(w_tgt)

fig, ax = plt.subplots(1, 2, figsize=(11.5, 4.3))

ax[0].plot(l_sgd, color="crimson", lw=1.9, label="SGD, $\\alpha$ = 1e-7 (largest stable)")
ax[0].plot(l_adam, color="tab:blue", lw=1.9, label="Adam, $\\alpha$ = 1e-3")
ax[0].axhline(target, color="0.3", ls="--", lw=1.2)
ax[0].annotate(f"optimum = {target:.3f}", xy=(2, target), xytext=(0, 7),
               textcoords="offset points", fontsize=8, color="0.3", ha="left")
ax[0].set_xlabel("epoch"); ax[0].set_ylabel("loss")
ax[0].set_title(f"One feature rescaled by 1000x  ($\\kappa$ = "
                f"{np.linalg.cond(X_bad.T @ X_bad / len(y)):.0e})", fontsize=9)
ax[0].legend(fontsize=8)

# --- right: the raw gradient spans decades; the Adam step does not.
gen = np.random.default_rng(0)
w_a = np.zeros(p)
m, v, t = np.zeros(p), np.zeros(p), 0
for _ in range(200):                    # warm the moment estimates up
    idx = gen.choice(len(y), 32, replace=False)
    g = batch_gradient(w_a, idx)
    t += 1
    m = 0.9 * m + 0.1 * g
    v = 0.999 * v + 0.001 * g ** 2
    step = 1e-3 * (m / (1 - 0.9 ** t)) / (np.sqrt(v / (1 - 0.999 ** t)) + 1e-8)
    w_a = w_a - step

j = np.arange(p)
ax[1].bar(j - 0.2, np.abs(g), width=0.4, color="crimson", label="$|g_j|$  (raw gradient)")
ax[1].bar(j + 0.2, np.abs(step), width=0.4, color="tab:blue",
          label="$|\\Delta w_j|$  (Adam's step)")
ax[1].set_yscale("log")
ax[1].axhline(1e-3, color="0.3", ls=":", lw=1.4, label=r"the step size $\alpha = 10^{-3}$")
ax[1].set_xlabel("coordinate $j$"); ax[1].set_ylabel("magnitude  (log scale)")
ax[1].set_title("Gradients span decades. The steps do not.", fontsize=9)
ax[1].set_xticks(j)
ax[1].legend(fontsize=8, loc="upper center")
ax[1].set_ylim(top=np.abs(g).max() * 60)

plt.tight_layout()
plt.show()

print(f"|g| spans   {np.abs(g).max() / np.abs(g).min():10.1f}x  across coordinates")
print(f"|step| spans{np.abs(step).max() / np.abs(step).min():10.1f}x  across coordinates")

X = X_orig                              # restore, again

**Figure 4 — Adam adapts to the scale, not to the landscape.**

*Left:* the same problem as the table above. SGD is pinned to $\alpha = 10^{-7}$ — anything
larger is unstable, because stability needs $\alpha < 2/L$ and rescaling one feature by
$1000$ multiplied $L$ by $10^6$. That step is then hopelessly small for the *other* nine
coordinates, so the run crawls. Adam reaches the optimum in a few epochs at
$\alpha = 10^{-3}$, four orders of magnitude larger.

*Right:* the mechanism, coordinate by coordinate, after 200 warm-up steps, on a log axis.
The red bars are the raw gradient components — coordinate 0 towers over the rest, exactly
as the rescaling dictates, and across all ten coordinates they span a factor of about
**37 000**. The blue bars are the steps Adam actually takes: they span a factor of about
**7**, and all sit near $\alpha$ (the dotted line). Dividing by $\sqrt{\hat v_j}$ has
absorbed the per-coordinate scale almost entirely.

> **Connect this to day 1.** Standardizing $X$ fixes this problem in the *data*; Adam fixes
> it in the *optimizer*. Both are answers to a large $\kappa$. Neither is a substitute for
> understanding that $\kappa$ was the problem — and note that Adam's fix is per-coordinate,
> so it handles badly scaled *axes* but not a bowl whose axes are rotated away from the
> coordinate system. Day 4's Newton handles that case, because it uses the full Hessian.

On the badly scaled problem SGD is still struggling where Adam has essentially finished.
Not because Adam is a cleverer descent method, but because it **stopped caring about the
scale of the coordinates**.

---

## 4. Epochs, shuffling, seeds

Three practical points that are not decoration.

**Epoch.** One pass through all $n$ samples, i.e. $n/b$ steps. Compare methods by *epoch*
(equal data cost), not by iteration — one SGD iteration and one gradient-descent
iteration are not remotely the same amount of work, and plotting against iteration count
flatters whichever method touches more data per step.

**Shuffling.** Reshuffle every epoch. Data often arrives sorted by label or by time, and
marching through it in order means each epoch applies the same *correlated* sequence of
biased updates. Shuffling is what makes the "uniformly drawn" assumption in section 1
roughly true.

**Seeds.** SGD is randomized, so two runs differ. Inject the generator
(`rng: np.random.Generator`) rather than calling `np.random.*` inside the optimizer. Then
a run is reproducible from its seed, a failing test can be replayed, and the randomness
is a *dependency* like any other — day 2's dependency inversion, applied to chance.

> **Design note.** `SGD` and `Adam` are new `Optimizer` implementations, not new
> `DirectionRule`s. Their loop genuinely differs — epochs, shuffling, no line search — so
> forcing them into `DescentOptimizer` would mean editing it. A new implementation of an
> existing interface is the open/closed answer. And both depend on `BatchObjective`
> alone: neither asks for a Hessian, a line search, or a full gradient. That is interface
> segregation doing real work.

---

## 5. Today's labs

| Lab | What | The point |
|---|---|---|
| 1 (45 min) | `n_samples`, `batch_gradient` on `GLMLoss` | write `gradient` as the batch over *all* indices — do not duplicate the formula |
| 2 (55 min) | `SGD` + schedules + momentum | `batch_size = n` must reproduce gradient descent exactly |
| 3 (45 min) | `Adam` | first step $\approx \alpha\,\mathrm{sign}(g)$; far fewer epochs on the ill-scaled problem |
| 4 (15 min) | batch-size / noise study | loss vs epoch for $b \in \{1, 32, 256, n\}$; wall-clock vs epochs |

**Three questions for the debrief:**

1. Why is the mini-batch gradient unbiased — and where exactly does the argument use
   uniform sampling?
2. Why does a constant step not reach the exact optimum, when gradient descent with a
   constant step does?
3. What does Adam adapt *to*? Name the day-1 concept it is compensating for.

> **Tomorrow.** We have used the slope (day 2) and a sample of the slope (today).
> Tomorrow we use the **curvature** — and get a method that converges in a handful of
> steps regardless of $\kappa$, at the price of solving a linear system each time. We
> will also discover that its Hessian is an object statisticians already have a name for.